In [ ]:
# ============================================================
# 0) Imports
# ============================================================
from datetime import datetime, timedelta
import requests
import pandas as pd
import xmltodict
import aiohttp
import asyncio
import time
import re
import warnings
warnings.filterwarnings("ignore")
from tqdm import tqdm
import nest_asyncio
nest_asyncio.apply()

user = 'xxxxxxxxx'
password = 'xxxxxxxxxxxxx'

print(f"Task started at {datetime.now()}")

Task started at 2026-10-20 13:55:26.308638


In [5]:
# ============================================================
# مسارات الملفات
# ============================================================
emails_path         = r"Y:\Amr Atef\VCC Payment Autosend\uploads\resources\Emails Update.xlsx"
all_invoices_path   = r"W:\Invoices KSA VAT\All Invoices KSA VAT Received.xlsx"
old_situation_path  = r"W:\Invoices KSA VAT\Full Situation 14-Oct-2026\full Situation 14-10-2026.xlsx"
output_path         = fr"C:\Users\int34\Desktop\full Situation {datetime.now().strftime('%d-%m-%Y')}.xlsx"

In [6]:
From = 20261013
To = 20261019

In [ ]:
# ============================================================
# API 1: getBookingList
# ============================================================
def get_booking_codes():
    url = "xxxxxxxxxxxxxxxxxxxxx"
    headers = {
        'content-type': 'text/xml',
        'SOAPAction': 'xxxxxxxxxx',
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:66.0) Gecko/20100101 Firefox/66.0",
        "Accept-Encoding": "*",
        "Connection": "keep-alive"
    }
    body = f"""<?xml version="1.0" encoding="utf-8"?>
        <soap:Envelope xmlns:xsi="xxxxxxxxxxxxxxxxxxxx" xmlns:xsd="xxxxxxxxxxxxxxxxxxxx" xmlns:soap="xxxxxxxxxxxxxxxxxx">
          <soap:Body>
            <getBookingList xmlns="http://juniper.es/">
              <user>{user}</user>
              <password>{password}</password>
              <EndTravelDateFrom>{From}</EndTravelDateFrom>
              <EndTravelDateTo>{To}</EndTravelDateTo>
            </getBookingList>
          </soap:Body>
        </soap:Envelope>"""
    while True:
        try:
            response = requests.post(url, data=body, headers=headers, timeout=60)
            break
        except Exception as e:
            print(f"⚠️ Retry API1: {e}")
            time.sleep(2)

    data = xmltodict.parse(response.text)
    try:
        booking_list = data['soap:Envelope']['soap:Body']['getBookingListResponse']['getBookingListResult']['wsResult']['BookingList']['Booking']
    except (KeyError, TypeError):
        print("❌ No BookingList في الـ Response!")
        return pd.DataFrame()

    if not isinstance(booking_list, list):
        booking_list = [booking_list]
    return pd.DataFrame(booking_list)


print("=" * 60)
print("API 1: جلب قائمة الحجوزات...")
listBookingsDF = get_booking_codes()
column_mapping = {
    '@Id': 'Id', '@Status': 'Status', '@BookingCode': 'BookingCode',
    '@BookingDate': 'BookingDate', '@LastModifiedDate': 'LastModifiedDate',
    '@AgencyRef': 'AgencyRef', '@TimeZone': 'TimeZone'
}
listBookingsDF = listBookingsDF.rename(columns=column_mapping)
listBookingsDF = listBookingsDF[listBookingsDF['Status'].str.contains("OK|CON|CAC|CAN", na=False)]
print(f"✅ API 1 Returned: {len(listBookingsDF)} Booking")
print("=" * 60)

API 1: جلب قائمة الحجوزات...
✅ API 1 Returned: 2013 Booking


In [ ]:
# ============================================================
# API 2: getBookings (تفاصيل كل حجز)
# ============================================================
Juniper_columns = [
    'BookingCode', 'BookingEmail', 'Line Id', 'Country', 'VCC', 'Hotel Name',
    'NetCostLine', 'ProductGroup', 'PassengerName', 'BookingDate',
    'BeginTravelDate', 'EndTravelDate', 'Remarks', 'Customer',
    'Internal Remarks', 'Currency', 'Supplier Id', 'Supplier Name'
]
Juniper_rows = []
failed_codes = []


async def GetBookingData(session, code, pbar, semaphore, retries=3, delay=2):
    url = "xxxxxxxxxxxxxxxxxxxxx"
    headers = {
        'content-type': 'text/xml',
        'SOAPAction': "xxxxxxxxxxxxxxxx",
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:66.0) Gecko/20100101 Firefox/66.0",
        "Accept-Encoding": "*",
        "Connection": "keep-alive"
    }
    body = f"""<?xml version="1.0" encoding="utf-8"?>
        <soap:Envelope xmlns:xsi="xxxxxxxxxxxxxxxxx" xmlns:xsd="xxxxxxxxxxxxxxxxxxxxx" xmlns:soap="xxxxxxxxxxxxxxxxxxx">
          <soap:Body>
            <getBookings xmlns="http://juniper.es/">
              <user>{user}</user>
              <password>{password}</password>
              <BookingCode>{code}</BookingCode>
            </getBookings>
          </soap:Body>
        </soap:Envelope>"""

    async with semaphore:
        for attempt in range(retries):
            try:
                async with session.post(url, data=body, headers=headers, timeout=60) as response:
                    response_text = await response.text()
                    response_text = re.sub(r'[\x00-\x08\x0b\x0c\x0e-\x1f]', '', response_text)
                    responseDict = xmltodict.parse(response_text)

                    try:
                        AllBookings = responseDict['soap:Envelope']['soap:Body']['getBookingsResponse']['getBookingsResult']['wsResult']['Bookings']['Booking']
                    except (KeyError, TypeError):
                        failed_codes.append(code)
                        pbar.update(1)
                        return

                    lines = AllBookings['Lines']['Line']
                    if not isinstance(lines, list):
                        lines = [lines]

                    for line in lines:
                        # الإيميل والـ remark
                        try:
                            roomlist = line.get('roomlist', {}).get('room', {})
                            if isinstance(roomlist, list):
                                mail = roomlist[0].get('bookingemail', None)
                                remark = roomlist[0].get('confirmationreference', '')
                            else:
                                mail = roomlist.get('bookingemail', None)
                                remark = roomlist.get('confirmationreference', '')
                        except Exception:
                            mail = None
                            remark = ''

                        # Supplier
                        supplier = line.get('Supplier', {})
                        if isinstance(supplier, list):
                            s_ids = [s.get('@Id', '') for s in supplier if isinstance(s, dict)]
                            s_names = [s.get('SupplierName', '') for s in supplier if isinstance(s, dict)]
                            supplier_id = ', '.join([str(x) for x in s_ids if x])
                            supplier_name = ', '.join([x for x in s_names if x])
                        elif isinstance(supplier, dict):
                            supplier_id = supplier.get('@Id', '')
                            supplier_name = supplier.get('SupplierName', '')
                        else:
                            supplier_id = ''
                            supplier_name = ''

                        holder = AllBookings.get('Holder', {}) or {}
                        customer = AllBookings.get('Customer', {}) or {}
                        zone = line.get('Zone', {}) or {}

                        row = [
                            AllBookings.get('@BookingCode', ''),
                            mail,
                            line.get('@LineDate', ''),
                            zone.get('country', ''),
                            line.get('VirtualCreditCardPayment', ''),
                            line.get('ServiceName', ''),            # Hotel Name
                            line.get('NetCostLine', 0),
                            line.get('ProductGroupName', ''),
                            (holder.get('NameHolder', '') or '') + ' ' + (holder.get('LastName', '') or ''),
                            (AllBookings.get('@BookingDate', '') or '')[:10],
                            (line.get('BeginTravelDate', '') or '')[:10],
                            (line.get('EndTravelDate', '') or '')[:10],
                            remark,
                            customer.get('Name', ''),
                            AllBookings.get('InRemarks', ''),
                            line.get('CostCurrency', ''),
                            supplier_id,
                            supplier_name
                        ]
                        Juniper_rows.append(row)

                    pbar.update(1)
                    return

            except Exception as e:
                if attempt < retries - 1:
                    await asyncio.sleep(delay)
                else:
                    print(f"❌ Booking Failed {code}: {type(e).__name__} - {e}")
                    failed_codes.append(code)
                    pbar.update(1)
                    return


async def main():
    bookingCodes = list(set(listBookingsDF['BookingCode'].to_list()))
    semaphore = asyncio.Semaphore(5)
    async with aiohttp.ClientSession() as session:
        tasks = []
        with tqdm(total=len(bookingCodes), desc="API 2: Processing") as pbar:
            for code in bookingCodes:
                tasks.append(asyncio.ensure_future(GetBookingData(session, code, pbar, semaphore)))
            await asyncio.gather(*tasks, return_exceptions=True)


print("=" * 60)
print("API 2:  Return Booking Details ...")
asyncio.run(main())
Juniper_df = pd.DataFrame(Juniper_rows, columns=Juniper_columns)
print(f"✅ API 2 Return: {len(Juniper_df)} Row")
print("=" * 60)

API 2:  Return Booking Details ...


API 2: Processing: 100%|███████████████████████████████████████████████████████████| 2013/2013 [07:28<00:00,  4.49it/s]

✅ API 2 Return: 2062 Row


In [ ]:
# ============================================================
# API 3: getProductGroups
# ============================================================
def getProductGroups():
    url = "xxxxxxxxxxxxxxxx"
    headers = {
        'content-type': 'text/xml',
        'SOAPAction': 'xxxxxxxxxxxxxxxxx',
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:66.0) Gecko/20100101 Firefox/66.0",
        "Accept-Encoding": "*",
        "Connection": "keep-alive"
    }
    body = f"""<?xml version="1.0" encoding="utf-8"?>
        <soap:Envelope xmlns:xsi="xxxxxxxxxxx" xmlns:xsd="xxxxxxxxxx" xmlns:soap="xxxxxxxxxxxxx">
          <soap:Body>
            <getProductGroups xmlns="http://juniper.es/">
              <user>{user}</user>
              <password>{password}</password>
            </getProductGroups>
          </soap:Body>
        </soap:Envelope>"""
    response = requests.post(url, data=body, headers=headers, timeout=60)
    data = xmltodict.parse(response.text)['soap:Envelope']['soap:Body']['getProductGroupsResponse']['getProductGroupsResult']['wsResult']['GroupProducts']['GroupProduct']
    if not isinstance(data, list):
        data = [data]
    return pd.DataFrame(data)


print("=" * 60)
print("API 3: Get Product Groups...")
productGroupsDF = getProductGroups()
print(f"✅ API 3 Returned: {len(productGroupsDF)} ProductGroup")
print("=" * 60)

API 3: Get Product Groups...
✅ API 3 Returned: 395 ProductGroup


In [10]:
# ============================================================
# merge مع ProductGroups
# ============================================================
Juniper_df = Juniper_df.merge(
    productGroupsDF[['Name', 'Department']],
    left_on='ProductGroup',
    right_on='Name',
    how='left'
)

In [11]:
# ============================================================
# الفلتر النهائي (بتاعك)
# ============================================================
Juniper_df = Juniper_df[~(Juniper_df['Internal Remarks'].str.contains('related|converted|issue', case=False, na=False))]
Juniper_df = Juniper_df[~(Juniper_df['Customer'].str.contains("How to work|Global International", case=False, na=False))]

departments = [
    'H2H - TDSTravel (JWS)',
    'ET Global (SA)',
    'H2H - DOTW',
    'HotelBeds',
    'HCD-Independent (Sabre)'
]

Juniper_df['NetCostLine'] = pd.to_numeric(Juniper_df['NetCostLine'], errors='coerce')

Juniper_df = Juniper_df[
    (Juniper_df['Department'].isin(departments)) &
    (Juniper_df['Country'] == 'Saudi Arabia') &
    (Juniper_df['VCC'] == 'False') &
    (Juniper_df['NetCostLine'] > 0)
]

Juniper_df = Juniper_df.drop(['Name'], axis=1)
Juniper_df = Juniper_df.reset_index(drop=True)
print(f"✅ بعد الفلتر النهائي: {len(Juniper_df)} حجز")
print("=" * 60)

✅ بعد الفلتر النهائي: 401 حجز


In [12]:
# ============================================================
# تجهيز Full (rename + split PassengerName)
# ============================================================
Juniper_df[['First Name', 'Last Name']] = Juniper_df['PassengerName'].str.split(' ', n=1, expand=True)

Juniper_df = Juniper_df.rename(columns={
    'Supplier Name': 'Supplier',
    'Country': 'Hotel Country',
    'NetCostLine': 'Net Cost',
    'Currency': 'Cost Currency',
    'BookingEmail': 'hotel email',
    'Remarks': 'Remarks'
})

In [13]:
# ============================================================
# Department_Map → Destination Managers
# ============================================================
Department_Map = {
    "ET Global (SA)": "(Mr.M.Fathy,Mr.Ahmed ElSayed)",
    "HotelBeds": "(Mr.M.Fathy,Mr.Ahmed ElSayed)",
    "H2H - TDSTravel (JWS)": "(Mr.M.Fathy,Mr.Ahmed ElSayed)",
    "H2H - DOTW": "(Mr.M.Fathy,Mr.Ahmed ElSayed)",
    "HCD-Independent (Sabre)": "(Mr.M.Fathy,Mr.Ahmed ElSayed)"
}
Juniper_df['Destination Managers'] = Juniper_df['Department'].map(Department_Map)

In [14]:
# ============================================================
# Country Group
# ============================================================
country_mapping = {'Egypt': 'Egypt', 'Saudi Arabia': 'KSA', 'United Arab Emirates': 'UAE'}
Juniper_df['Country Group'] = Juniper_df['Hotel Country'].map(country_mapping).fillna('International')

In [15]:
# ============================================================
# merge مع emails
# ============================================================
emails = pd.read_excel(emails_path)
emails['Lower Hotel Name'] = emails['Accommodation Name'].apply(lambda x: x.lower() if pd.notna(x) else x)
Juniper_df['Lower Hotel Name'] = Juniper_df['Hotel Name'].apply(lambda x: x.lower() if pd.notna(x) else x)
Juniper_df = Juniper_df.merge(emails[['Lower Hotel Name', 'Email']], on='Lower Hotel Name', how='left')

In [16]:
# drop_duplicates على Line Id
Juniper_df = Juniper_df.drop_duplicates(subset='Line Id')

In [17]:
# fillna email
Juniper_df['Email'] = Juniper_df['Email'].fillna(Juniper_df['hotel email'])

In [18]:
# remove our domains
ourdomains = ['eetglobal', 'egyptexpresstvl', 'etg.sa', 'mztourism']
def remove_our_emails(email):
    if not isinstance(email, (str, bytes)):
        return ''
    hotels_emails = []
    found = re.findall(r"\b[\w.-]+@[\w.-]+\.[\w.-]+\b", email)
    for i in found:
        if i.split('@')[1].split('.')[0] not in ourdomains:
            hotels_emails.append(i)
    return ';'.join(hotels_emails)

Juniper_df['Email'] = Juniper_df['Email'].apply(remove_our_emails)
Juniper_df['hotel email'] = Juniper_df['Email']
Juniper_df = Juniper_df.drop(columns=['Email', 'Lower Hotel Name'])

In [19]:
# ============================================================
# اختيار الأعمدة النهائية للـ Juniper_df (بدون Received)
# ============================================================
Juniper_df = Juniper_df[[
    'BookingCode', 'Hotel Name', 'Supplier', 'Hotel Country', 'Country Group',
    'Destination Managers', 'hotel email', 'First Name', 'Last Name',
    'Cost Currency', 'Net Cost', 'BeginTravelDate', 'EndTravelDate', 'Remarks'
]]

In [20]:
# ============================================================
# دمج مع Full القديم
# ============================================================
oldfull_df = pd.read_excel(old_situation_path, sheet_name='Full', engine='openpyxl')
newfull_df = pd.concat([oldfull_df, Juniper_df], axis=0, ignore_index=True)

In [21]:
# ============================================================
# Received/Not Received (بعد الدمج عشان يحدث القديم)
# ============================================================
all_invoices_df = pd.read_excel(all_invoices_path)
received_codes = set(all_invoices_df['Booking code'].astype(str))

newfull_df['BookingCode'] = newfull_df['BookingCode'].astype(str)
newfull_df['Received/Not Received'] = newfull_df['BookingCode'].apply(
    lambda x: 'received' if x in received_codes else 'not received'
)

In [22]:
# ============================================================
# ترتيب الأعمدة + تنظيف
# ============================================================
columns_to_clean = ['Hotel Name', 'Supplier', 'Hotel Country', 'Country Group', 'Destination Managers']
for column in columns_to_clean:
    newfull_df[column] = newfull_df[column].astype(str).str.replace(r'\s+', ' ', regex=True).str.strip()

desired_order = [
    'BookingCode', 'Hotel Name', 'Supplier', 'Hotel Country', 'Country Group',
    'Destination Managers', 'hotel email', 'Received/Not Received',
    'First Name', 'Last Name', 'Cost Currency', 'Net Cost',
    'BeginTravelDate', 'EndTravelDate', 'Remarks'
]
newfull_df = newfull_df[desired_order]

In [23]:
# drop_duplicates على BookingCode (نخلي الأحدث)
newfull_df['BookingCode'] = newfull_df['BookingCode'].astype(str)
newfull_df = newfull_df.drop_duplicates(subset='BookingCode', keep='last').reset_index(drop=True)

In [24]:
# format dates
def format_date(date_obj):
    if pd.isna(date_obj):
        return date_obj
    if isinstance(date_obj, datetime):
        return date_obj.strftime('%Y-%m-%d')
    elif isinstance(date_obj, str):
        for fmt in ('%m/%d/%Y', '%m/%d/%y', '%d/%m/%Y', '%d/%m/%y', '%Y-%m-%d', '%d-%m-%Y'):
            try:
                return datetime.strptime(date_obj, fmt).strftime('%Y-%m-%d')
            except ValueError:
                continue
    return date_obj

newfull_df['EndTravelDate'] = newfull_df['EndTravelDate'].apply(format_date)
newfull_df['BeginTravelDate'] = newfull_df['BeginTravelDate'].apply(format_date)

In [25]:
# lowercase Hotel Name & Supplier
newfull_df['Hotel Name'] = newfull_df['Hotel Name'].apply(lambda x: x.lower() if pd.notna(x) else x)
newfull_df['Supplier'] = newfull_df['Supplier'].apply(lambda x: x.lower() if pd.notna(x) else x)

In [26]:
# ============================================================
# Contracts sheet
# ============================================================
newContracts_df = newfull_df[['Supplier', 'Destination Managers']].drop_duplicates(subset=['Supplier', 'Destination Managers']).copy()

try:
    oldContracts_df = pd.read_excel(old_situation_path, sheet_name='Contracts', engine='openpyxl')
    for column in ['Supplier', 'Destination Managers']:
        oldContracts_df[column] = oldContracts_df[column].astype(str).str.replace(r'\s+', ' ', regex=True).str.strip()

    newContracts_df['merge'] = newContracts_df['Supplier'] + newContracts_df['Destination Managers']
    oldContracts_df['merge'] = oldContracts_df['Supplier'] + oldContracts_df['Destination Managers']

    newContracts_df = newContracts_df.merge(oldContracts_df[['merge', 'Chain']], on='merge', how='left').fillna('None')
    newContracts_df = newContracts_df.merge(oldContracts_df[['merge', 'Contract With', 'Contract Situation ']], on='merge', how='left').fillna('Pending Contracting')
    newContracts_df = newContracts_df.merge(oldContracts_df[['merge', 'Notes']], on='merge', how='left').fillna('-')
    newContracts_df = newContracts_df.drop_duplicates(subset=['Supplier', 'Destination Managers'])
    newContracts_df = newContracts_df.drop(columns=['merge'])
except Exception as e:
    print(f"⚠️ مشكلة في Contracts: {e}")
    newContracts_df['Chain'] = 'None'
    newContracts_df['Contract With'] = 'Pending Contracting'
    newContracts_df['Contract Situation '] = 'Pending Contracting'
    newContracts_df['Notes'] = '-'

In [27]:
# ============================================================
# Summary sheet
# ============================================================
for column in columns_to_clean:
    newfull_df[column] = newfull_df[column].astype(str).str.replace(r'\s+', ' ', regex=True).str.strip()

Requested_df = newfull_df.groupby(
    by=['Hotel Name', 'Supplier', 'Destination Managers', 'Hotel Country', 'Country Group']
).count().reset_index()
Requested_df = Requested_df[['Hotel Name', 'Supplier', 'Destination Managers', 'Hotel Country', 'Country Group', 'BookingCode']]
Requested_df = Requested_df.rename(columns={'BookingCode': 'Requested'})
Requested_df['merge_column'] = (Requested_df['Hotel Name'] + Requested_df['Supplier'] +
                                Requested_df['Destination Managers'] + Requested_df['Hotel Country'] +
                                Requested_df['Country Group'])
Requested_df = Requested_df.drop_duplicates(subset=['merge_column'])

filtered_df = newfull_df[newfull_df['Received/Not Received'].str.contains('not', case=False, na=False) == False]
Received_df = filtered_df.groupby(
    by=['Hotel Name', 'Supplier', 'Destination Managers', 'Hotel Country', 'Country Group']
).count().reset_index()
Received_df = Received_df[['Hotel Name', 'Supplier', 'Destination Managers', 'Hotel Country', 'Country Group', 'BookingCode']]
Received_df = Received_df.rename(columns={'BookingCode': 'Received'})
Received_df['merge_column'] = (Received_df['Hotel Name'] + Received_df['Supplier'] +
                               Received_df['Destination Managers'] + Received_df['Hotel Country'] +
                               Received_df['Country Group'])
Received_df = Received_df.drop_duplicates(subset=['merge_column'])

Requested_df = Requested_df.merge(Received_df[['merge_column', 'Received']], on='merge_column', how='left')
Requested_df['Received'] = Requested_df['Received'].fillna(0)
Requested_df['Not Received'] = Requested_df['Requested'] - Requested_df['Received']

In [28]:
# merge مع Summary القديم
try:
    oldsummary_df = pd.read_excel(old_situation_path, sheet_name='Summary', engine='openpyxl')
    oldsummary_df['Supplier'] = oldsummary_df['Supplier'].apply(lambda x: x.lower() if pd.notna(x) else x)
    for column in columns_to_clean:
        oldsummary_df[column] = oldsummary_df[column].astype(str).str.replace(r'\s+', ' ', regex=True).str.strip()
    oldsummary_df['merge_column'] = (oldsummary_df['Hotel Name'] + oldsummary_df['Supplier'] +
                                     oldsummary_df['Destination Managers'] + oldsummary_df['Hotel Country'] +
                                     oldsummary_df['Country Group'])
    Requested_df = Requested_df.merge(
        oldsummary_df[['merge_column', 'Invoices Directed to', 'Invoices Notes']],
        on='merge_column', how='left'
    ).fillna({'Invoices Directed to': 'Not Received', 'Invoices Notes': 'Not Received'})
except Exception as e:
    print(f"⚠️ مشكلة في Summary merge: {e}")
    Requested_df['Invoices Directed to'] = 'Not Received'
    Requested_df['Invoices Notes'] = 'Not Received'

Requested_df = Requested_df.drop_duplicates(subset=['Hotel Name', 'Supplier', 'Hotel Country', 'Country Group', 'Destination Managers'])
Requested_df = Requested_df.drop(columns=['merge_column'])

In [29]:
# merge مع Contracts
Requested_df['merged_column'] = Requested_df['Supplier'] + Requested_df['Destination Managers']
newContracts_df['merged_column'] = newContracts_df['Supplier'] + newContracts_df['Destination Managers']
Requested_df = Requested_df.merge(
    newContracts_df[['merged_column', 'Contract With', 'Contract Situation ']],
    on='merged_column', how='left'
)
Requested_df.rename(columns={'Contract Situation ': 'Contract Notes'}, inplace=True)
Requested_df['Contract Notes'] = Requested_df['Contract Notes'].fillna('Pending Contracting')
Requested_df['Contract and Invoices Notes'] = (Requested_df['Contract Notes'].astype(str) + "," +
                                               Requested_df['Invoices Notes'].astype(str) + " Invoices")
Requested_df = Requested_df.drop(columns=['merged_column'])

Requested_df = Requested_df[[
    'Hotel Name', 'Supplier', 'Destination Managers', 'Hotel Country', 'Country Group',
    'Requested', 'Received', 'Not Received', 'Invoices Directed to',
    'Contract With', 'Invoices Notes', 'Contract Notes', 'Contract and Invoices Notes'
]]

In [30]:
# ============================================================
# الحفظ النهائي
# ============================================================
with pd.ExcelWriter(output_path, engine='xlsxwriter') as writer:
    newfull_df.to_excel(writer, sheet_name='Full', index=False)
    Requested_df.to_excel(writer, sheet_name='Summary', index=False)
    newContracts_df.to_excel(writer, sheet_name='Contracts', index=False)

print("=" * 60)
print(f"✅ تم حفظ الملف في: {output_path}")
print(f"   - Full:      {len(newfull_df)} حجز")
print(f"   - Summary:   {len(Requested_df)} صف")
print(f"   - Contracts: {len(newContracts_df)} صف")
print(f"⏰ Task finished at {datetime.now()}")

✅ تم حفظ الملف في: C:\Users\int34\Desktop\full Situation 20-10-2026.xlsx
   - Full:      32510 حجز
   - Summary:   1701 صف
   - Contracts: 404 صف
⏰ Task finished at 2026-10-20 14:03:32.613804
